## Task

Task from the IT-Jim Computer Vision course.

Detect a planar marker in the scene using ORB keypoints and BFMatcher, estimate a homography between the marker image and video frame, and draw the projected marker region in each frame.

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

# Load marker and video
img = cv2.imread('../source/marker.jpg', cv2.IMREAD_GRAYSCALE)
cap = cv2.VideoCapture('../source/find_chocolate.mp4')

# Create ORB detector with 1000 keypoints with a scaling pyramid factor of 1.2, nlevel
orb = cv2.ORB_create(1500, 1.3, 13)    

# Detect keypoints of original image
kp_m, des_m = orb.detectAndCompute(img, None)

# For video output
fps = 20
frame_width = int(cap.get(3))
frame_height = int(cap.get(4))
out = cv2.VideoWriter('ORB_Homography.avi',cv2.VideoWriter_fourcc('M','J','P','G'), 10, (frame_width,frame_height))

# Capture frame-by-frame
while (cap.isOpened()):      
    ret, frame = cap.read()
    if ret == True:
        g_frame = cv2.cvtColor(frame,cv2.COLOR_BGR2GRAY)
        # Detect keypoints of rotated image
        kp_r, des_r = orb.detectAndCompute(g_frame, None)

        # Draw Keypoints in each frame 
        #frame = cv2.drawKeypoints(frame, kp_s, None)   

        # Do matching 
        bf = cv2.BFMatcher(cv2.NORM_HAMMING, crossCheck=True)
        matches = bf.match(des_m,des_r)
        # Draw Matches 
        #frame = cv2.drawMatches(img, kp_m, frame, kp_s, matches[:], None, 
        #                        flags=cv2.DrawMatchesFlags_NOT_DRAW_SINGLE_POINTS)   

        # Select the matches based on distance
        selected = []
        for m in matches:
            if m.distance < 50:
                selected.append(m) 

        # Number matches limit
        if len(selected) > 30:
            # Extract the matched keypoints
            src_pts = np.float32([ kp_m[m.queryIdx].pt for m in selected ]).reshape(-1,1,2)
            dst_pts = np.float32([ kp_r[m.trainIdx].pt for m in selected ]).reshape(-1,1,2)
        
            # Find homography
            hm, mask = cv2.findHomography(src_pts, dst_pts, cv2.RANSAC, 5.0)
            matchesMask = mask.ravel().tolist()
            h,w = img.shape
            pts = np.float32([[0,0],[0,h-1],[w-1,h-1],[w-1,0]]).reshape(-1,1,2)
            dst = cv2.perspectiveTransform(pts, hm)
        
            # Draw found region        
            res = cv2.polylines(frame, [np.int32(dst)], True,255,3, cv2.LINE_AA)
        else:
            print('Not enought features')

        # Write video
        out.write(res)

    else:
        break

print("all done")

# Release all
cap.release()
out.release()
cv2.destroyAllWindows()